# Band Power Ratios

**Dataset**: PhysioNet Auditory EEG  
**Channels**: P4, Cz, F8, T7  
**Sampling rate**: 200 Hz  
**Subject**: 1

---

## Overview

We compute band power ratios as mental state indicators: Theta/Alpha for stress, Alpha/Beta for relaxation, and Theta/Beta for attention.

## Expected outputs

- Three plots showing the three ratios over time
- Dashed horizontal line representing the mean for each ratio
- Fluctuation reflecting changing mental state over time

## Key parameters

| Parameter | Value | Meaning |
| --- | --- | --- |
| WINDOW | 1000 | 5 second window |
| STEP | 500 | 2.5 second step |
| Bands | 4 | Delta, Theta, Alpha, Beta |


## 1. Install dependencies


In [ ]:
!pip install scipy numpy plotly mne wfdb


## 2. Clone repo and download data

We download only subject 1 (`--subjects 1`) to speed up the experiment in Colab.


In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')


In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.dat')):
    !python data/download_local.py --output data/local --subjects 1


## 3. Load the EEG signal

We load subject 1, experiment 1, session 2.


In [ ]:
import numpy as np
from utils.eeg_loader import load_local_eeg

timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=1, experiment=1, session=2
)
fs = 200

print(f'Channels: {ch_names}')
print(f'Signal length: {len(eeg_data)} samples ({len(eeg_data)/fs:.1f} seconds)')


## 4. Compute band power ratios

We compute band powers on sliding windows, then derive the three ratios.


In [ ]:
from scipy.signal import welch

WINDOW = 1000
STEP = 500
BANDS = {
    'delta': (0.5, 4),
    'theta': (4, 8),
    'alpha': (8, 13),
    'beta': (13, 30),
}

def compute_band_powers(segment, fs):
    freqs, psd = welch(segment, fs=fs, nperseg=1024)
    powers = {}
    for name, (fmin, fmax) in BANDS.items():
        mask = (freqs >= fmin) & (freqs <= fmax)
        powers[name] = np.trapezoid(psd[mask], freqs[mask])
    return powers

channel_data = eeg_data[:, 0]
n_windows = (len(channel_data) - WINDOW) // STEP + 1
theta_alpha = []
alpha_beta = []
theta_beta = []
window_centers = []

for i in range(n_windows):
    start = i * STEP
    end = start + WINDOW
    segment = channel_data[start:end]
    powers = compute_band_powers(segment, fs)
    theta_alpha.append(powers['theta'] / powers['alpha'])
    alpha_beta.append(powers['alpha'] / powers['beta'])
    theta_beta.append(powers['theta'] / powers['beta'])
    window_centers.append((start + end) / 2 / fs)

theta_alpha = np.array(theta_alpha)
alpha_beta = np.array(alpha_beta)
theta_beta = np.array(theta_beta)
print(f'Computed {n_windows} windows')


## 5. Interactive plot

**What to look for:**

- Windows deviating from the mean indicate mental state changes
- High Theta/Alpha suggests stress, low suggests relaxation
- High Alpha/Beta suggests relaxation, low suggests attention



In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    subplot_titles=('Stress indicator (Theta/Alpha)',
                                    'Relaxation indicator (Alpha/Beta)',
                                    'Attention indicator (Theta/Beta)'))
fig.add_trace(go.Scatter(x=window_centers, y=theta_alpha, name='Theta/Alpha',
                         line=dict(color='red', width=1.5)), row=1, col=1)
fig.add_hline(y=np.mean(theta_alpha), line_dash='dash', line_color='black', row=1, col=1)
fig.add_trace(go.Scatter(x=window_centers, y=alpha_beta, name='Alpha/Beta',
                         line=dict(color='green', width=1.5)), row=2, col=1)
fig.add_hline(y=np.mean(alpha_beta), line_dash='dash', line_color='black', row=2, col=1)
fig.add_trace(go.Scatter(x=window_centers, y=theta_beta, name='Theta/Beta',
                         line=dict(color='blue', width=1.5)), row=3, col=1)
fig.add_hline(y=np.mean(theta_beta), line_dash='dash', line_color='black', row=3, col=1)
fig.update_layout(height=900, title_text='Band Power Ratios - Mental State Indicators',
                  xaxis3_title='Time (s)', showlegend=True)
fig.show()


## What did we learn?

- Band ratios are simple yet effective mental state indicators
- Theta/Alpha for stress, Alpha/Beta for relaxation, Theta/Beta for attention
- Used as per-individual baselines in practice
- No machine learning needed but less accurate for complex tasks

